# Notebook 06 — Mini-Net vs Fine-Tuned YOLOv7 Comparison
**Part 2: Side-by-side analysis of scratch-trained GymDetectorMini vs fine-tuned YOLOv7**  
**Team:** Varun Gazala · Mohit Raiyani · Jatinkumar Nabhoya · UNH Spring 2026

**Prerequisites:** `05_mininet_train.ipynb` must have been run (produces `results/mininet_metrics.json`).

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import json, os, sys, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import torch

sys.path.insert(0, '..')

CLASS_NAMES  = ['dumbbell', 'barbell', 'kettlebell', 'resistance_band', 'pull_up_bar']
COLORS       = ['#e63946', '#2a9d8f', '#457b9d', '#e9c46a', '#8338ec']
ANCHORS_PATH = '../mininet/anchors.json'
MINI_W       = '../weights/mininet_best.pt'
YOLOv7_W     = '../best.pt'
DATASET_DIR  = '../dataset_augmented'
IMG_SIZE     = 320
STRIDES      = [8, 16, 32]

# ── Load results JSONs ────────────────────────────────────────────────────────
MINI_JSON = '../results/mininet_metrics.json'
YOLOv7_JSON = '../results/augmented_metrics.json'

assert os.path.exists(MINI_JSON), \
    f'Run 05_mininet_train.ipynb first — {MINI_JSON} not found'

with open(MINI_JSON) as f:
    mini = json.load(f)

# Part-1 results (hardcoded fallback if JSON missing)
yolo_default = {
    'map_50': 0.4603, 'map': 0.2392,
    'per_class_ap50': {
        'dumbbell': 0.1747, 'barbell': 0.0731, 'kettlebell': 0.7000,
        'resistance_band': 0.1067, 'pull_up_bar': 0.0980
    },
    'n_params': 36_501_466
}
if os.path.exists(YOLOv7_JSON):
    with open(YOLOv7_JSON) as f:
        yolo = json.load(f)
    # Merge in n_params if not stored
    yolo.setdefault('n_params', yolo_default['n_params'])
else:
    yolo = yolo_default
    print(f'Note: {YOLOv7_JSON} not found — using hardcoded Part-1 results')

print('Results loaded.')
print(f'  Mini-Net  — mAP@0.5={mini["map_50"]:.4f}, params={mini.get("n_params",0):,}')
print(f'  YOLOv7    — mAP@0.5={yolo["map_50"]:.4f}, params={yolo.get("n_params",0):,}')

In [ ]:
# ── Section 1: Headline Comparison Table ─────────────────────────────────────
from IPython.display import display, HTML

mini_p  = mini.get('n_params', 0)
yolo_p  = yolo.get('n_params', 36_501_466)
mini_mb = mini_p * 4 / 1e6    # float32 approx
yolo_mb = yolo_p * 4 / 1e6

rows = [
    ('Parameters',      f'{mini_p:,}',              f'{yolo_p:,}',
     f'mini = {mini_p/yolo_p*100:.1f}%'),
    ('Model size (MB)', f'~{mini_mb:.0f} MB',       f'~{yolo_mb:.0f} MB',
     f'{yolo_mb/mini_mb:.0f}× larger'),
    ('mAP@0.5',         f'{mini["map_50"]:.4f}',    f'{yolo["map_50"]:.4f}',
     f'Δ = {mini["map_50"]-yolo["map_50"]:+.4f}'),
    ('mAP@0.5:0.95',    f'{mini["map"]:.4f}',       f'{yolo["map"]:.4f}',
     f'Δ = {mini["map"]-yolo["map"]:+.4f}'),
]

html = '<table border="1" style="border-collapse:collapse;font-family:monospace">'
html += '<tr><th>Metric</th><th>Mini-Net (scratch)</th><th>YOLOv7 fine-tuned</th><th>Δ / ratio</th></tr>'
for row in rows:
    html += '<tr>' + ''.join(f'<td style="padding:6px">{c}</td>' for c in row) + '</tr>'
html += '</table>'
display(HTML(html))

In [ ]:
# ── Section 2: Per-Class AP@0.5 Bar Chart ────────────────────────────────────
mini_ap50  = mini.get('per_class_ap50', {})
yolo_ap50  = yolo.get('per_class_ap50', {})

x   = np.arange(len(CLASS_NAMES))
w   = 0.35
mini_vals = [mini_ap50.get(c, 0.0) for c in CLASS_NAMES]
yolo_vals = [yolo_ap50.get(c, 0.0) for c in CLASS_NAMES]

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - w/2, mini_vals, w, label='Mini-Net (scratch)', color='#457b9d', alpha=0.85)
bars2 = ax.bar(x + w/2, yolo_vals, w, label='YOLOv7 fine-tuned',  color='#e63946', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([c.replace('_', '\n') for c in CLASS_NAMES], fontsize=10)
ax.set_ylabel('AP@0.5', fontsize=12)
ax.set_title('Per-Class AP@0.5 — Mini-Net vs Fine-Tuned YOLOv7', fontsize=13)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../docs/figures/comparison_per_class_ap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 3: Mini-Net Training Curves ──────────────────────────────────────
hist = mini.get('training_history', {})
if hist:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(hist.get('train_loss', []), label='Train Loss', color='#2a9d8f')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title('Mini-Net Training Loss'); ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(hist.get('val_map50', []), label='Val mAP@0.5', color='#e63946')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('mAP@0.5')
    ax2.set_title('Mini-Net Val mAP@0.5'); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('../docs/figures/mininet_loss_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No training history in metrics JSON — skipping curve plot.')

In [ ]:
# ── Section 4: Side-by-Side Predictions on Test Images ───────────────────────
# Requires both model checkpoints to be present.
import random
from mininet.model   import GymDetectorMini
from mininet.anchors import load_anchors
from mininet.dataset import _letterbox
from mininet.utils   import decode_all, nms_single, draw_boxes_cv2

DEVICE = 'cuda' if torch.cuda.is_available() else \
         'mps'  if torch.backends.mps.is_available() else 'cpu'

# ── Load Mini-Net ──
mini_model = None
if os.path.exists(MINI_W) and os.path.exists(ANCHORS_PATH):
    ANCHORS    = load_anchors(ANCHORS_PATH)
    mini_model = GymDetectorMini().to(DEVICE)
    mini_model.load_state_dict(torch.load(MINI_W, map_location=DEVICE))
    mini_model.eval()
    print(f'Mini-Net loaded from {MINI_W}')
else:
    print(f'Mini-Net weights not found — skipping prediction comparison')

# ── Load Fine-Tuned YOLOv7 ──
yolo_model = None
YOLOV7_DIR = '../yolov7'
if os.path.exists(YOLOv7_W) and os.path.exists(YOLOV7_DIR):
    sys.path.insert(0, YOLOV7_DIR)
    try:
        import torch.nn as nn
        from models.experimental import attempt_load
        from utils.general        import non_max_suppression, scale_coords
        from utils.datasets       import letterbox as yolo_letterbox
        from utils.torch_utils    import select_device

        COCO_W = os.path.join(YOLOV7_DIR, 'yolov7.pt')
        yolo_model = attempt_load(COCO_W, map_location=DEVICE)
        detect = yolo_model.model[-1]
        na_ = detect.na; new_no = 5 + 5
        for i, conv in enumerate(detect.m):
            detect.m[i] = nn.Conv2d(conv.in_channels, new_no*na_, 1).to(DEVICE)
        detect.nc = 5; detect.no = new_no
        yolo_model.nc = 5; yolo_model.names = CLASS_NAMES
        yolo_model.load_state_dict(torch.load(YOLOv7_W, map_location=DEVICE))
        yolo_model.eval()
        print(f'YOLOv7 fine-tuned loaded from {YOLOv7_W}')
        YOLO_AVAILABLE = True
    except Exception as e:
        print(f'Could not load YOLOv7: {e}')
        YOLO_AVAILABLE = False
else:
    YOLO_AVAILABLE = False
    print('YOLOv7 weights / yolov7/ not found — skipping YOLOv7 predictions')


def run_mininet(img_rgb, conf=0.25):
    img_lb, ratio, (pl, pt) = _letterbox(img_rgb, IMG_SIZE)
    img_t = torch.from_numpy(img_lb).permute(2,0,1).float().unsqueeze(0).to(DEVICE)/255.0
    with torch.no_grad():
        raw  = mini_model(img_t)
        pred = decode_all(raw, ANCHORS, STRIDES)[0]
        dets = nms_single(pred, conf_thres=conf, iou_thres=0.45).cpu()
    if len(dets):
        h0, w0 = img_rgb.shape[:2]
        dets[:,0] = ((dets[:,0]-pl)/ratio).clamp(0, w0)
        dets[:,1] = ((dets[:,1]-pt)/ratio).clamp(0, h0)
        dets[:,2] = ((dets[:,2]-pl)/ratio).clamp(0, w0)
        dets[:,3] = ((dets[:,3]-pt)/ratio).clamp(0, h0)
    return draw_boxes_cv2(img_rgb, dets.numpy().astype(int)) if len(dets) else img_rgb.copy()


def run_yolov7(img_rgb, conf=0.25):
    img_lb, _, _ = yolo_letterbox(img_rgb, 640, auto=False, scaleFill=False)
    img_t = torch.from_numpy(img_lb).float().permute(2,0,1).unsqueeze(0).to(DEVICE)/255.0
    with torch.no_grad():
        raw = yolo_model(img_t)[0]
    dets = non_max_suppression(raw, conf_thres=conf, iou_thres=0.45)[0]
    if dets is not None and len(dets):
        dets[:,:4] = scale_coords(img_t.shape[2:], dets[:,:4], img_rgb.shape).round()
        d = dets.cpu().numpy()
        rows = [(CLASS_NAMES[int(r[5])], r[4], *map(int, r[:4])) for r in d]
        return draw_boxes_cv2(img_rgb, rows)
    return img_rgb.copy()


# ── Plot side-by-side ──
if mini_model is not None:
    test_imgs = sorted(
        glob.glob(f'{DATASET_DIR}/images/test/*.jpg') +
        glob.glob(f'{DATASET_DIR}/images/test/*.jpeg') +
        glob.glob(f'{DATASET_DIR}/images/test/*.png')
    )
    show_paths = test_imgs[:6]

    cols = 3 if YOLO_AVAILABLE else 2
    fig, axes = plt.subplots(len(show_paths), cols,
                             figsize=(5*cols, 4*len(show_paths)))
    if len(show_paths) == 1:
        axes = axes[None]

    for i, path in enumerate(show_paths):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        axes[i][0].imshow(img); axes[i][0].set_title('GT Image', fontsize=9); axes[i][0].axis('off')
        axes[i][1].imshow(run_mininet(img)); axes[i][1].set_title('Mini-Net', fontsize=9); axes[i][1].axis('off')
        if YOLO_AVAILABLE:
            axes[i][2].imshow(run_yolov7(img)); axes[i][2].set_title('YOLOv7 FT', fontsize=9); axes[i][2].axis('off')

    plt.suptitle('Predictions: Original | Mini-Net | YOLOv7 fine-tuned', fontsize=13)
    plt.tight_layout()
    plt.savefig('../docs/figures/comparison_predictions.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved → docs/figures/comparison_predictions.png')

In [ ]:
# ── Section 5: Summary Discussion ─────────────────────────────────────────────
delta_map50 = mini['map_50'] - yolo['map_50']
param_ratio = mini.get('n_params', 0) / yolo.get('n_params', 36_501_466)

print('=' * 60)
print('COMPARISON SUMMARY')
print('=' * 60)
print(f'  Parameter ratio : Mini-Net is {param_ratio*100:.1f}% of YOLOv7')
print(f'  mAP@0.5 gap     : {delta_map50:+.4f} (mini − yolo)')
print()
print('Key observations:')
print('  1. Mini-Net uses ~10x fewer params but the mAP gap reflects')
print('     the massive advantage of YOLOv7\'s COCO pretraining on')
print('     a 648-image dataset.')
print('  2. Pretraining transfers far more information than any')
print('     architecture efficiency gain at this data scale.')
print('  3. Mini-Net wins on: model size, inference speed, edge deployability.')
print('  4. YOLOv7 wins on: all accuracy metrics, especially hard classes')
print('     (barbell, resistance_band) that need rich low-level features.')